# Real Misclassification & Deep Error Analysis

This notebook critically examines the exact, true failure modes of the optimal InceptionV3 model based on its verified validation subset outputs. Analysis is rooted exclusively in generated JSON artifacts.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 6)

## 1. Interpretating the True Verified Confusion Matrix

In [ ]:
cm_path = 'inception_79%/confusion_matrix.json'
if os.path.exists(cm_path):
    with open(cm_path, 'r') as f:
        cm = np.array(json.load(f))
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Abnormal', 'Normal'])
    disp.plot(cmap='Reds', values_format='d')
    plt.title('True Validation Confusion Matrix')
    plt.grid(False)
    plt.show()
else:
    print("Real confusion matrix JSON not found. Cannot proceed safely.")

## 2. In-Depth Error Pattern Analysis (Real Results)

According to the true confusion outputs: `[[284, 64], [113, 387]]`

### A. False Positive Risk (113 Cases)
- **What happened**: 113 healthy (Normal) images were incorrectly flagged as Abnormal.
- **Clinical Consequence**: This induces patient anxiety and requires a secondary manual review by an ophthalmologist. While highly inefficient for medical throughput, this does NOT carry a severe clinical danger (no diseased patients missed).
- **Why it happens**: Normal fundus images occasionally contain bright artifacts (like optic disc highlights or photo glare) that convolutions mistakenly associate with hard exudates or cotton wool spots.

### B. False Negative Risk (64 Cases)
- **What happened**: 64 pathological (Abnormal) images were passed off as entirely Normal.
- **Clinical Consequence**: **Extremely High Risk**. These patients have Diabetic Retinopathy but will be discharged without treatment, potentially leading to irreversible blindness and malpractice exposure.
- **Why it happens**: Mild Non-Proliferative Diabetic Retinopathy (NPDR) usually only presents with microaneurysms that are notoriously tiny and blend into the background. The aggressive image augs / zooming may have cropped these out, or the resolution constraints (224x224) eroded the pixel-level micro-vascular cues.

### C. The Hardest Class
The data proves **Abnormal is significantly harder to perfectly capture with high precision**, but the model defaults to being highly sensitive (*conservative*) and categorizing ambiguities as Abnormal safely, which forces high Recall for Abnormal (81.6%) and accounts for our 113 False Positives.

## 3. Data-Driven Next Steps for Real-World Iteration

1. **Asymmetric Cost Function Deployment**: Since False Negatives are clinically devastating, we must introduce a custom Loss Function (e.g. Weighted Binary Cross-Entropy) where penalizing a False Negative is intrinsically 5x more expensive than a False Positive. This would push those 64 FN cases severely down.
2. **Spatial Attention Mechanisms**: The traditional Global Average Pooling throws out exact spatial coordinates. By utilizing a transformer branch or precise bounding-box detection, the network wouldn't "lose" extreme minute microaneurysms.